In [ ]:
!pip install -q 'torch==2.8.0' 'torch_xla[tpu]==2.8.0' transformers


In [ ]:

import os, time
os.environ['PJRT_DEVICE'] = 'TPU'   # PJRT runtime (PyTorch/XLA 2.0+), NOT XRT_TPU_CONFIG
t0=time.time()
import torch
print('torch', torch.__version__)
import torch_xla
print('torch_xla', torch_xla.__version__)
import torch_xla.core.xla_model as xm
dev = xm.xla_device()
print('XLA device:', dev, ' count:', xm.xrt_world_size())
print('setup done in %.1fs' % (time.time()-t0))


In [ ]:

import time
from transformers import AutoModelForCausalLM, AutoConfig
t0=time.time()
cfg = AutoConfig.from_pretrained('Qwen/Qwen3.5-0.8B')
print('0.8B arch:', cfg.model_type, ' layers:', cfg.num_hidden_layers)
d = cfg.to_dict()
for k in ('linear_attn','layer_types','gdn','hybrid'):
    if k in d:
        print('  cfg['+k+'] =', str(d[k])[:120])
m08 = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3.5-0.8B', torch_dtype=torch.float32)
print('loaded 0.8B in %.1fs' % (time.time()-t0))
m08 = m08.to(dev)   # -> xla
import torch_xla
torch_xla.sync()
t1=time.time()
x = torch.zeros(1, 16, dtype=torch.long, device=dev)
attn = torch.ones(1, 16, dtype=torch.long, device=dev)
out = m08(x, attention_mask=attn)     # first forward = XLA compile (slow)
torch_xla.sync()
print('0.8B FIRST forward (compile) out', out.logits.shape, 'in %.1fs' % (time.time()-t1))
t2=time.time()
_ = m08(x, attention_mask=attn)       # cached re-run
torch_xla.sync()
print('0.8B second forward %.3fs' % (time.time()-t2))
print('GDN_08B_OK')


In [ ]:

import time
# minimal autograd backward through the same GDN path (our Jacobian relies on per-token grads)
t0=time.time()
x2 = torch.zeros(1, 16, dtype=torch.long, device=dev)
m08.requires_grad_(True)
out2 = m08(x2, attention_mask=attn)
loss = out2.logits.sum()
loss.backward()
torch_xla.sync()
grads = [p.grad for p in m08.parameters() if p.grad is not None]
print('backward OK in %.1fs; n_params_with_grad=%d' % (time.time()-t0, len(grads)))
print('BWD_OK')


In [ ]:

import time
t0=time.time()
m4 = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3.5-4B', torch_dtype=torch.float32)
print('loaded 4B in %.1fs' % (time.time()-t0))
m4 = m4.to(dev)
torch_xla.sync()
t1=time.time()
x4 = torch.zeros(1, 16, dtype=torch.long, device=dev)
out4 = m4(x4, attention_mask=attn)
torch_xla.sync()
print('4B FIRST forward (compile) out', out4.logits.shape, 'in %.1fs' % (time.time()-t1))
print('GDN_4B_OK')


In [ ]:

print('=================================')
print('TPU CANARY: all *_OK printed above => Qwen3.5 GDN compiles on XLA v3-8. TPU is viable.')
print('=================================')
